# M2 Final Evaluation
**Hybrid Music Recommendation System — DSCI 441, Spring 2026**

Evaluates four models on a shared held-out test set:
- `PopularityBaseline` — non-personalized floor
- `CF-only` — ALS (implicit 0.7.2, factors=64)
- `Content-only` — k-NN cosine on 9 audio features, seeded by user's most-played training song
- `Hybrid` — adaptive-alpha blend of CF + Content

Metrics: Precision, Recall, NDCG, HitRate @ K=5, 10, 20  
All aggregates reported with bootstrap 95% CIs (1000 resamples).  
Hypothesis tests use paired bootstrap (10 000 resamples) on NDCG@10.

**Methodology note:**  
The ALS model was trained on the full interaction matrix before the evaluation split.
For CF and Hybrid recommendations, we construct a *training-only* user row (80% of
the user's interactions) so held-out items are not filtered by `filter_already_liked_items`.
This is the standard pragmatic protocol for retrospective ALS evaluation in class projects.
The user's latent factor still encodes all interactions (mild upward bias); it is noted
in the report.

---
## Section 1 — Setup

In [ ]:
import os, sys
os.environ['OPENBLAS_NUM_THREADS'] = '1'

from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
from scipy.sparse import csr_matrix

from src.data import (
    load_msd_triplets, load_msd_metadata, load_spotify_tracks,
    match_datasets, build_metadata_catalog,
)
from src.models import CFModel, ContentModel, PopularityBaseline
from src.hybrid import HybridRecommender
from src.evaluation import (
    evaluate_model, bootstrap_ci, paired_bootstrap_test,
    precision_at_k, recall_at_k, ndcg_at_k, hit_rate_at_k,
)

MODELS_DIR   = Path('../models')
RESULTS_DIR  = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

K_VALUES = [5, 10, 20]

In [ ]:
# Load CF model (pre-trained ALS from M1/Block 1)
cf = CFModel.load(MODELS_DIR)
print(f'CF: {len(cf._user_to_idx):,} users, {len(cf._idx_to_song):,} songs')

# Build song_to_idx once (used in training-only row construction later)
cf_song_to_idx = {sid: idx for idx, sid in cf._idx_to_song.items()}

In [ ]:
# Load MSD + Spotify, build metadata catalog
msd_meta  = load_msd_metadata()
spotify   = load_spotify_tracks()
matched   = match_datasets(spotify, msd_meta)
meta_cat  = build_metadata_catalog(matched)
print(f'matched: {len(matched):,} rows, {matched["song_id"].nunique():,} unique song_ids')
print(f'metadata_catalog: {len(meta_cat):,} song_ids')

In [ ]:
# Load Content model (pre-fit k-NN on dedup'd 7,611 tracks)
cm = ContentModel.load(MODELS_DIR)
print(f'Content k-NN index: {len(cm._songid_to_idx):,} songs')
print(f'CF ∩ Content overlap: {len(set(cf._idx_to_song.values()) & set(cm._songid_to_idx)):,} songs')

In [ ]:
# Fit PopularityBaseline on the same filtered triplets used to train CF
_TRIPLETS_CACHE = RESULTS_DIR / 'triplets_cache.pkl'
if _TRIPLETS_CACHE.exists():
    triplets = joblib.load(_TRIPLETS_CACHE)
    print('Loaded triplets from cache')
else:
    triplets = load_msd_triplets()
    joblib.dump(triplets, _TRIPLETS_CACHE)
    print('Loaded and cached triplets')

print(f'Triplets: {len(triplets):,} rows')

pop = PopularityBaseline().fit(triplets)
print('PopularityBaseline fit OK')

In [ ]:
# Build HybridRecommender
hybrid = HybridRecommender(cf, cm, meta_cat, alpha_strategy='adaptive')
print(f'HybridRecommender ready, overlap={len(hybrid._overlap_ids):,} songs')

---
## Section 2 — Test Split Construction

In [ ]:
_SPLIT_CACHE = RESULTS_DIR / 'eval_split.pkl'

if _SPLIT_CACHE.exists():
    split = joblib.load(_SPLIT_CACHE)
    sampled_users          = split['sampled_users']
    test_data              = split['test_data']
    user_train_songs       = split['user_train_songs']
    user_most_played_in_ct = split['user_most_played_in_ct']
    print(f'Loaded split from cache ({len(sampled_users)} users)')
else:
    rng = np.random.default_rng(42)
    all_users = list(cf._user_to_idx.keys())

    # Every user in the CF model has >= 20 unique songs (MIN_USER_COUNT filter),
    # so all are eligible for the >= 10 interaction requirement.
    sampled_users = rng.choice(all_users, size=1000, replace=False).tolist()

    test_data        = {}   # user_id -> set(held_out_song_ids)
    user_train_songs = {}   # user_id -> list(training_song_ids)

    for uid in sampled_users:
        uidx     = cf._user_to_idx[uid]
        song_idxs = cf._user_item[uidx].indices          # column indices with plays
        song_ids  = np.array([cf._idx_to_song[i] for i in song_idxs])

        n_test   = max(1, int(len(song_ids) * 0.20))
        perm     = rng.permutation(len(song_ids))

        test_data[uid]        = set(song_ids[perm[:n_test]])
        user_train_songs[uid] = list(song_ids[perm[n_test:]])

    # For each user, find their most-played training song that's in the content index.
    # Uses the sparse row directly to avoid densifying 98K-dim vectors.
    user_most_played_in_ct = {}   # user_id -> song_id (or absent if none found)
    for uid in sampled_users:
        uidx     = cf._user_to_idx[uid]
        row      = cf._user_item[uidx]
        row_dict = dict(zip(row.indices, row.data))

        best_sid, best_val = None, -1.0
        for sid in user_train_songs[uid]:
            if sid in cm._songid_to_idx and sid in cf_song_to_idx:
                val = row_dict.get(cf_song_to_idx[sid], 0.0)
                if val > best_val:
                    best_sid, best_val = sid, val
        if best_sid is not None:
            user_most_played_in_ct[uid] = best_sid

    joblib.dump({
        'sampled_users':          sampled_users,
        'test_data':              test_data,
        'user_train_songs':       user_train_songs,
        'user_most_played_in_ct': user_most_played_in_ct,
    }, _SPLIT_CACHE)
    print('Built and cached split')

n_users_with_content_seed = len(user_most_played_in_ct)
print(f'\nSplit stats:')
print(f'  Total sampled users:           {len(sampled_users)}')
print(f'  Users with a content seed:     {n_users_with_content_seed} '
      f'({100*n_users_with_content_seed/len(sampled_users):.1f}%)')
test_sizes = [len(v) for v in test_data.values()]
print(f'  Held-out items per user:       '
      f'median={np.median(test_sizes):.0f}, '
      f'min={min(test_sizes)}, max={max(test_sizes)}')

# Save canonical user list so cold_start_evaluation.ipynb loads
# the EXACT same 1000 users -- required for a valid warm vs cold comparison.
joblib.dump(sampled_users, RESULTS_DIR / 'test_users.pkl')
print('Saved test_users.pkl')

---
## Section 3 — Run All 4 Models

In [ ]:
# ── recommend_fn definitions ──────────────────────────────────────────────────
# Each function: (user_id: str, k: int) -> list[str]  (ordered song_ids)

def cf_fn(user_id, k):
    """
    Build a training-only user row so held-out songs are not filtered.
    CF's user embedding was trained with all interactions (mild upward bias),
    but at least the held-out songs can surface as candidates.
    """
    uidx   = cf._user_to_idx[user_id]
    full   = cf._user_item[uidx].toarray().flatten()
    mask   = np.zeros(full.shape, dtype=bool)
    for sid in user_train_songs[user_id]:
        if sid in cf_song_to_idx:
            mask[cf_song_to_idx[sid]] = True
    train_row = csr_matrix(full * mask)
    idxs, _   = cf._model.recommend(uidx, train_row, N=k,
                                     filter_already_liked_items=True)
    return [cf._idx_to_song[int(i)] for i in idxs]


def content_fn(user_id, k):
    """Content-only: seed from user's most-played training song in content index."""
    seed = user_most_played_in_ct.get(user_id)
    if seed is None:
        return []   # user has no training songs in content catalog
    return cm.recommend(seed, k=k)['song_id'].tolist()


def pop_fn(user_id, k):
    """Non-personalized: globally most-played songs."""
    return pop.recommend(k=k)['song_id'].tolist()


def hybrid_fn(user_id, k):
    """
    Hybrid with training-only known_song_ids so held-out items are not filtered
    from the CF component.
    """
    recs = hybrid.recommend(
        user_id=user_id, k=k,
        known_song_ids=set(user_train_songs[user_id]),
    )
    return recs['song_id'].tolist()

In [ ]:
_METRICS_CACHE = RESULTS_DIR / 'per_user_metrics.csv'

if _METRICS_CACHE.exists():
    per_user = pd.read_csv(_METRICS_CACHE)
    print(f'Loaded per-user metrics from cache ({len(per_user):,} rows)')
else:
    model_fns = {
        'popularity': pop_fn,
        'cf':         cf_fn,
        'content':    content_fn,
        'hybrid':     hybrid_fn,
    }

    dfs = []
    for name, fn in model_fns.items():
        print(f'Evaluating {name}...', end=' ', flush=True)
        df = evaluate_model(fn, test_data, k_values=K_VALUES)
        df['model'] = name
        dfs.append(df)
        print(f'done ({len(df):,} rows)')

    per_user = pd.concat(dfs, ignore_index=True)
    per_user.to_csv(_METRICS_CACHE, index=False)
    print(f'Saved per_user_metrics.csv ({len(per_user):,} rows)')

---
## Section 4 — Aggregate with Bootstrap 95% CIs

**Stop here** and review these numbers before running Sections 5–6.

In [ ]:
_CI_CACHE = RESULTS_DIR / 'final_metrics.csv'

if _CI_CACHE.exists():
    final_metrics = pd.read_csv(_CI_CACHE)
    print('Loaded final_metrics from cache')
else:
    rows = []
    for (model, k, metric), grp in per_user.groupby(['model', 'k', 'metric']):
        point, lo, hi = bootstrap_ci(grp['value'].values, n_resamples=1000, seed=42)
        rows.append({
            'model':   model,
            'k':       k,
            'metric':  metric,
            'mean':    round(point, 4),
            'ci_low':  round(lo,    4),
            'ci_high': round(hi,    4),
        })
    final_metrics = pd.DataFrame(rows)
    final_metrics.to_csv(_CI_CACHE, index=False)
    print('Saved final_metrics.csv')

In [ ]:
# Pretty-print as markdown for each metric at K=10
MODEL_ORDER  = ['popularity', 'cf', 'content', 'hybrid']
METRIC_ORDER = ['precision', 'recall', 'ndcg', 'hit_rate']

for metric in METRIC_ORDER:
    subset = (
        final_metrics
        [final_metrics['metric'] == metric]
        .sort_values(['k', 'model'])
    )
    print(f'\n### {metric.upper()}@K')
    print(f'| model | K=5 | K=10 | K=20 |')
    print(f'|---|---|---|---|')
    for model in MODEL_ORDER:
        row_vals = []
        for k in K_VALUES:
            r = subset[(subset['model'] == model) & (subset['k'] == k)]
            if len(r):
                r = r.iloc[0]
                row_vals.append(f"{r['mean']:.4f} [{r['ci_low']:.4f}, {r['ci_high']:.4f}]")
            else:
                row_vals.append('—')
        print(f'| {model} | ' + ' | '.join(row_vals) + ' |')

In [ ]:
# Full table (all K, all metrics)
pivot = final_metrics.pivot_table(
    index=['model', 'k'],
    columns='metric',
    values='mean'
).round(4)
pivot = pivot.reindex(columns=METRIC_ORDER)
print(pivot.to_string())

---
## ⛔ STOP HERE — review results above before running Section 5+

Check:
1. Hybrid NDCG@10 > CF-only NDCG@10 (if not, investigate alpha or overlap)
2. CF >> Popularity (baseline sanity check)
3. All CIs are non-overlapping where expected

If results look wrong, **do not proceed** — investigate before running hypothesis tests.

---
## Section 5 — Hypothesis Tests (paired bootstrap, 10 000 resamples)

In [ ]:
_HT_CACHE = RESULTS_DIR / 'hypothesis_tests.csv'

if _HT_CACHE.exists():
    ht_results = pd.read_csv(_HT_CACHE)
    print('Loaded hypothesis tests from cache')
else:
    # Extract per-user NDCG@10 for each model
    def get_ndcg10(model_name):
        sub = per_user[
            (per_user['model'] == model_name) &
            (per_user['k'] == 10) &
            (per_user['metric'] == 'ndcg')
        ].set_index('user_id')['value']
        return sub.reindex(sampled_users).fillna(0).values

    ndcg_hybrid  = get_ndcg10('hybrid')
    ndcg_pop     = get_ndcg10('popularity')
    ndcg_cf      = get_ndcg10('cf')
    ndcg_content = get_ndcg10('content')

    comparisons = [
        ('hybrid', 'popularity', ndcg_hybrid, ndcg_pop),
        ('hybrid', 'cf',         ndcg_hybrid, ndcg_cf),
        ('hybrid', 'content',    ndcg_hybrid, ndcg_content),
    ]

    ht_rows = []
    for model_a, model_b, scores_a, scores_b in comparisons:
        p = paired_bootstrap_test(scores_a, scores_b, n_resamples=10_000, seed=42)
        delta = scores_a.mean() - scores_b.mean()
        ht_rows.append({
            'model_a': model_a,
            'model_b': model_b,
            'metric':  'ndcg@10',
            'delta':   round(delta, 4),
            'p_value': round(p, 4),
            'significant_p05': p < 0.05,
        })
        print(f'  {model_a} vs {model_b}: Δ={delta:+.4f}, p={p:.4f}')

    ht_results = pd.DataFrame(ht_rows)
    ht_results.to_csv(_HT_CACHE, index=False)
    print('\nSaved hypothesis_tests.csv')

ht_results

---
## Section 6 — Visualizations

In [ ]:
# ── Figure 1: NDCG@10 bar chart with 95% CI error bars ───────────────────────
ndcg10 = final_metrics[
    (final_metrics['metric'] == 'ndcg') & (final_metrics['k'] == 10)
].set_index('model')

models  = MODEL_ORDER
means   = [ndcg10.loc[m, 'mean']   for m in models]
ci_low  = [ndcg10.loc[m, 'ci_low'] for m in models]
ci_high = [ndcg10.loc[m, 'ci_high'] for m in models]
errs_lo = [m - lo for m, lo in zip(means, ci_low)]
errs_hi = [hi - m for m, hi in zip(means, ci_high)]

fig, ax = plt.subplots(figsize=(7, 4))
colors  = ['#4c72b0', '#dd8452', '#55a868', '#c44e52']
bars    = ax.bar(models, means, color=colors, alpha=0.85,
                  yerr=[errs_lo, errs_hi], capsize=5,
                  error_kw={'elinewidth': 1.5, 'ecolor': 'black'})
ax.set_ylabel('NDCG@10')
ax.set_title('NDCG@10 across models (bootstrap 95% CI)', fontsize=12)
ax.set_ylim(0, max(ci_high) * 1.3)

for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)

fig.tight_layout()
fig.savefig(RESULTS_DIR / 'comparison_ndcg10_bar.png', dpi=150)
plt.show()
print('Saved comparison_ndcg10_bar.png')

In [ ]:
# ── Figure 2: each metric vs K, all 4 models ─────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
axes      = axes.flatten()
line_styles = {m: s for m, s in zip(MODEL_ORDER, ['-o', '-s', '-^', '-D'])}
palette     = {m: c for m, c in zip(MODEL_ORDER, colors)}

for ax, metric in zip(axes, METRIC_ORDER):
    sub = final_metrics[final_metrics['metric'] == metric]
    for model in MODEL_ORDER:
        mdf = sub[sub['model'] == model].sort_values('k')
        ax.plot(mdf['k'], mdf['mean'],
                line_styles[model], label=model,
                color=palette[model], linewidth=1.8, markersize=5)
        ax.fill_between(mdf['k'], mdf['ci_low'], mdf['ci_high'],
                         alpha=0.12, color=palette[model])
    ax.set_title(metric.upper() + '@K', fontsize=11)
    ax.set_xlabel('K')
    ax.set_xticks(K_VALUES)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    ax.legend(fontsize=8)

fig.suptitle('Ranking metrics vs K — all 4 models (shading = 95% CI)', fontsize=12)
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'comparison_metrics_vs_k.png', dpi=150)
plt.show()
print('Saved comparison_metrics_vs_k.png')